# Migration to GRIB2 - practical 3
## Vertical levels: `levtype=hl` and `levtype=sol`

In GRIB1, many fields that are really "a parameter at a given level" are encoded as separate **surface**
parameters: `100u` for the u-wind at 100 m, `stl1` ... `stl4` for the soil temperature in the four soil layers.

In GRIB2 the **parameter** and the **level** are kept apart: **one paramId for all levels**, with the level
given by `levelist`.

| Case | What changes | Parameters |
| --- | --- | --- |
| 1 | **nothing**: levels at 10 m and below stay in `levtype=sfc` | 2 m temperature, 10 m wind |
| 2 | heights **above 10 m** move to **`levtype=hl`** (height levels) | 100 m and 200 m wind |
| 3 | soil layers move to **`levtype=sol`** (surface other levels) | soil temperature, volumetric soil moisture |

All data come from **MARS**, for the same forecast, **1 September 2026 00 UTC**:  
- Mix of GRIB1 and GRIB2 (depending on the parameter) with `expver=1`, and
- the GRIB2 test data with **`expver=2250`**.

`expver` changes only because the GRIB2 data are a test dataset. Once GRIB2 is operational, it will be
`expver=1` again.

`grid` and `area` interpolate the data to 0.25° over Europe, to keep the downloads small.

## Setup

You need **earthkit-data**, **earthkit-plots** and a recent **ecCodes under the hood**.  
ecCodes older than 2.42.0 cannot decode the new encoding correctly - always use the latest release.  

In [ ]:
import earthkit.data as ekd

# The keys we print for every field
keys = ["metadata.edition", "metadata.shortName", "metadata.paramId", "metadata.levtype",
        "metadata.typeOfLevel", "metadata.level", "metadata.levelist"]

## Levels at 10 m and below: no change

Parameters at 2 m and 10 m above ground - such as 2 m temperature, 2 m dewpoint, 10 m wind - are **exempt**.
They keep their paramId and stay in `levtype=sfc`, without `levelist`.

### GRIB1

In [ ]:
near_sfc_grib1 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/near_sfc_grib1.grib"
).to_fieldlist()

In [ ]:
near_sfc_grib1.ls(keys=keys)

### GRIB2

In [ ]:
near_sfc_grib2 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/near_sfc_grib2.grib"
).to_fieldlist()

In [ ]:
near_sfc_grib2.ls(keys=keys)

| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `param` | 167/165/166 | 167/165/166 | same |
| `levtype` | `sfc` | `sfc` | same |
| `levelist` | - | - | same |
| level in the data | `surface`, 0 | `heightAboveGround`, 2 or 10 | changed |

The request stays the same. Only the encoding inside the file is more precise: GRIB2 says the field is 2 m or
10 m above ground.

## Heights above 10 m: `levtype=hl`

### GRIB1

Four paramIds, one per parameter and height.

In [ ]:
wind_grib1 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/uv_grib1.grib"
).to_fieldlist()

In [ ]:
wind_grib1.ls(keys=keys)

Note that the GRIB1 metadata says `surface`, level 0: the height (100 m, 200 m) is only known from the
paramId.

### GRIB2

Two paramIds - the ordinary `u` and `v` - and the height in `levelist`.

In [ ]:
wind_grib2 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/uv_grib2.grib"
).to_fieldlist()

In [ ]:
wind_grib2.ls(keys=keys)

The values are the same:

In [ ]:
wind_grib1.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.maximum"])

In [ ]:
wind_grib2.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.levelist", "metadata.maximum"])

| GRIB1 `param` (`levtype=sfc`) | GRIB2 `param` | GRIB2 `levtype` | GRIB2 `levelist` |
| --- | --- | --- | --- |
| 228246 (`100u`) | 131 (`u`) | `hl` | 100 |
| 228247 (`100v`) | 132 (`v`) | `hl` | 100 |
| 228239 (`200u`) | 131 (`u`) | `hl` | 200 |
| 228240 (`200v`) | 132 (`v`) | `hl` | 200 |
| 228249 (`100si`) | 10 (`ws`) | `hl` | 100 |
| 228241 (`200si`) | 10 (`ws`) | `hl` | 200 |

> **Watch out:** `u` at 100 m now has the same paramId (131) as `u` on pressure levels or model levels.
> Selecting by paramId or shortName alone is no longer enough - always use the **level type** and level too.

The old paramIds are **not** available in GRIB2. Asking for `100u` with `levtype=sfc` returns no data:

In [ ]:
try:
    ekd.from_source(
        "mars",
        {
            "class": "od",
            "stream": "oper",
            "type": "fc",
            "expver": "2250",
            "date": "20260901",
            "time": "00",
            "levtype": "sfc",
            "param": "228246",        # old paramId: 100u
            "timespan": "none",
            "step": "24",
            "grid": "0.25/0.25",
            "area": "75/-30/25/45",
        },
    ).to_fieldlist()
except Exception as e:
    print("Request failed:", e)

## Soil layers: `levtype=sol`

### GRIB1

Eight paramIds: soil temperature and volumetric soil water, one per layer.

In [ ]:
soil_grib1 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/soil_grib1.grib"
).to_fieldlist()

In [ ]:
soil_grib1.ls(keys=keys + ["metadata.maximum"])

### GRIB2

Two paramIds - `sot` and `vsw` - and the layer in `levelist`.

In [ ]:
soil_grib2 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/soil_grib2.grib"
).to_fieldlist()

In [ ]:
soil_grib2.ls(keys=keys + ["metadata.maximum"])

| GRIB1 `param` (`levtype=sfc`) | GRIB2 `param` | GRIB2 `levtype` | GRIB2 `levelist` | layer depth |
| --- | --- | --- | --- | --- |
| 139 / 170 / 183 / 236 (`stl1` ... `stl4`) | 260360 (`sot`) | `sol` | 1 / 2 / 3 / 4 | |
| 39 / 40 / 41 / 42 (`swvl1` ... `swvl4`) | 260199 (`vsw`) | `sol` | 1 / 2 / 3 / 4 | |
| | | | 1 | 0-7 cm |
| | | | 2 | 7-28 cm |
| | | | 3 | 28-100 cm |
| | | | 4 | 100-289 cm |

Same values, but look at `level`: in GRIB1 it is the **top of the layer in cm** (0, 7, 28, 100), in GRIB2 it is
the **layer number** (1, 2, 3, 4). Code that uses `level` to find a soil layer needs to change.

## Your turn now

### Practical A
Modify the requests below to download the wind speed at 100 m and 200 m (`100si`, `200si`). (You can find the
paramIds [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Parametersatheightsaboveground))

In [ ]:
ws_grib1 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/si_grib1.grib"
).to_fieldlist()

In [ ]:
ws_grib2 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/si_grib2.grib"
).to_fieldlist()

In [ ]:
ws_grib1.ls(keys=keys)

In [ ]:
ws_grib2.ls(keys=keys)

### Practical B
Modify the requests below to download the sea-ice temperature in the four ice layers (`istl1` ... `istl4`).
(You can find the paramIds [here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-Soiltemperature,volumetricsoilmoistureandsea-icetemperature))

Compare the two. What is the `typeOfLevel` in GRIB2?

In [ ]:
ice_grib1 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/sea_ice_t_grib1.grib"
).to_fieldlist()

In [ ]:
ice_grib2 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/opencharts-sample-data/2026-data/sea_ice_t_grib2.grib"
).to_fieldlist()

In [ ]:
ice_grib1.ls(keys=keys)

In [ ]:
ice_grib2.ls(keys=keys)

Further reading:
- [Migration to GRIB2 - Test GRIB2 data in MARS](https://confluence.ecmwf.int/pages/viewpage.action?pageId=716086411)
- [Changes in the MARS language](https://confluence.ecmwf.int/pages/viewpage.action?pageId=621051006)
- [Migration to GRIB2 - changes to encoding of parameters](https://confluence.ecmwf.int/pages/viewpage.action?pageId=554148197)